# 05. Integration: End-to-End Static Sudoku Vision & Solution Pipeline

### Overview
This notebook executes the complete end-to-end Sudoku pipeline on a static image:
1. **Load Image**
2. **Detect Grid Corners**
3. **Perspective Warp (900×900)**
4. **CNN Digit Recognition**
5. **Backtracking Solve**
6. **Augmented Reality Unwarping Overlay** (casting the green solution digits back onto the original angled photo)

In [ ]:
import os
import sys
from pathlib import Path

def find_root() -> Path:
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / "models" / "digit_classifier.keras").exists():
            return parent
    return Path(os.path.abspath("..")).resolve()

PROJECT_ROOT = find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from backend.cv_pipeline import (
    find_grid_corners,
    warp_grid,
    split_cells,
    overlay_solution,
    unwarp_overlay
)
from backend.model_service import SudokuDigitRecognizer
from backend.solver_service import solve_sudoku

plt.rcParams['figure.dpi'] = 110
print("Pipeline modules loaded successfully.")

## 1. Load Model & Sample Image

In [ ]:
recognizer = SudokuDigitRecognizer(str(PROJECT_ROOT / "models" / "digit_classifier.keras"))
print(f"CNN Model Loaded: {recognizer.is_loaded}")

sample_path = PROJECT_ROOT / "data" / "raw" / "sample1.png"
if not sample_path.exists():
    sample_path = PROJECT_ROOT / "data" / "test_set" / "sample1.png"

img = cv2.imread(str(sample_path))
print(f"Processing image: {sample_path.name} | Shape: {img.shape}")

## 2. Corner Detection & Rectification

In [ ]:
corners = find_grid_corners(img)
if corners is None:
    raise ValueError("Grid corners could not be detected.")

warped, M = warp_grid(img, corners, size=900)
warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
cells = split_cells(warped_gray, margin=8)

print(f"Detected 4 corners. Rectified grid to 900x900. Extracted {len(cells)} cells.")

## 3. Digit Recognition via CNN

In [ ]:
original_grid, confidences, _ = recognizer.recognize_sudoku(cells)

detected_digits_count = sum(1 for row in original_grid for val in row if val != 0)
mean_confidence = np.mean([c for row in confidences for c in row])

print(f"Recognized {detected_digits_count} non-empty digits with {mean_confidence*100:.1f}% mean confidence.")

## 4. Backtracking Solver & AR Projection

In [ ]:
solve_success, solved_grid, nodes, time_ms, _ = solve_sudoku(original_grid)
print(f"Puzzle Solved: {solve_success} in {time_ms:.2f} ms ({nodes} nodes explored)")

# 1. Overlay solution on 900x900 warped image
warped_solution = overlay_solution(warped, original_grid, solved_grid)

# 2. Unwarp overlay onto original camera photo
augmented_solution = unwarp_overlay(img, warped_solution, M)

## 5. Visual Dashboard: The Complete Pipeline

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title("1. Original Photo Input")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(warped_solution, cv2.COLOR_BGR2RGB))
axes[1].set_title("2. Rectified Solved Grid (900×900)")
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(augmented_solution, cv2.COLOR_BGR2RGB))
axes[2].set_title("3. Augmented Reality Solution Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()